In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import paths as P

In [ ]:
TEST = {
    "DPA" : [35, 114],
    "DPB" : [42, 121],
    "DQA" : [29, 110],
    "DQB" : [45, 119],
    "DRA" : [30, 109],
    "DRB" : [42, 121]
}

import pandas as pd

df = pd.read_csv(os.path.join(P.MHC_SRC, "HLA2_IMGT.csv"))

# slice HLA_Seq on the rows whose HLA_Name matches

In [ ]:
# collected results
sliced_data = []

# one keyword at a time
for keyword, (start, end) in TEST.items():
    matched_rows = df[df["HLA_Name"].str.contains(keyword, case=False, na=False)].copy()

    if not matched_rows.empty:
        # cut the sequence
        matched_rows["Sliced_Seq"] = matched_rows["HLA_Seq"].str.slice(start-1, end)
        matched_rows["Keyword"] = keyword  # provenance
        sliced_data.append(matched_rows[["HLA_Name", "Sliced_Seq", "Keyword"]])
    else:
        print(f"No match found for keyword: {keyword}")

# combine
result_df = pd.concat(sliced_data, ignore_index=True)
result_df = result_df.drop (columns=["Keyword"])

# save
output_path = "HLA2_IMGT_MSA_sliced.csv"

result_df.to_csv(output_path, index=False)


print(f"Sliced sequences saved to: {output_path}")

result_df

In [ ]:
result_df['Sliced_Seq'] = result_df['Sliced_Seq'].str.replace("*", "")
result_df.to_csv('HLA2_IMGT_MSA_sliced_clean.csv', index=False)

In [ ]:
import pandas as pd

# The same alignment cell 1 reads. It was staged here once with '.' rewritten to '*', so both
# gap characters have to go to get the ungapped sequence.
df = pd.read_csv(os.path.join(P.MHC_SRC, "HLA2_IMGT.csv"))
df['HLA_Seq'] = df['HLA_Seq'].str.replace(r'[.*]', '', regex=True)
df.to_csv('HLA2_IMGT_MSA_clean.csv', index=False)

In [ ]:

import pandas as pd

df_1 = pd.read_csv(os.path.join(P.WORK, "mhc", "HLA2_IMGT_MSA_clean.csv"))
df_2 = pd.read_csv(os.path.join(P.WORK, "mhc", "HLA2_IMGT_MSA_sliced_clean.csv"))
# drop HLA-DRB2*01:01, HLA-DRB8*01:01
df_1 = df_1[df_1["HLA_Name"] != "HLA-DRB2*01:01"]
df_2 = df_2[df_2["HLA_Name"] != "HLA-DRB2*01:01"]
# drop HLA-DRB8*01:01
df_1 = df_1[df_1["HLA_Name"] != "HLA-DRB8*01:01"]
df_2 = df_2[df_2["HLA_Name"] != "HLA-DRB8*01:01"]
df_1 = df_1.sort_values(by=["HLA_Name"])
df_2 = df_2.sort_values(by=["HLA_Name"])

df_1 = df_1.reset_index(drop=True)
df_2 = df_2.reset_index(drop=True)

def get_start_idx(row):
    try:
        return row['HLA_Seq'].index(row['Sliced_Seq'])
    except ValueError:
        return -1  # not a substring of the full sequence

def get_end_idx(row):
    try:
        return row['HLA_Seq'].index(row['Sliced_Seq']) + len(row['Sliced_Seq'])
    except ValueError:
        return -1

# join the two frames by position
merged_df = pd.DataFrame({
    "HLA_Name": df_1["HLA_Name"],
    "HLA_Seq": df_1["HLA_Seq"],
    "Sliced_Seq": df_2["Sliced_Seq"]
})

# compute the indices
merged_df["start_idx"] = merged_df.apply(get_start_idx, axis=1)
merged_df["end_idx"] = merged_df.apply(get_end_idx, axis=1)

# check
merged_df = merged_df.sort_values(by=["HLA_Name"])
merged_df = merged_df.reset_index(drop=True)
merged_df
merged_df.to_csv("HLA2_IMGT_MSA_idx.csv", index=False)

In [ ]:
merged_df

In [ ]:
# add the sliced-sequence column
merged_df = merged_df.drop(columns=["Sliced_Seq"])
merged_df["Sliced_Seq"] = merged_df.apply(
    lambda row: row["HLA_Seq"][row["start_idx"]:row["end_idx"]] if row["start_idx"] >= 0 else "",
    axis=1
)
# check
merged_df

In [ ]:
merged_df['tmp'] = (
    merged_df['HLA_Seq'].astype(str) + '|' +
    merged_df['start_idx'].astype(str) + '|' +
    merged_df['end_idx'].astype(str)
)
merged_df

In [ ]:
merged_df["HLA_Seq_len"] = merged_df["HLA_Seq"].str.len()
merged_df["Sliced_Seq_len"] = merged_df["Sliced_Seq"].str.len()
print("HLA_Seq length distribution:")
print(merged_df["HLA_Seq_len"].value_counts().sort_index())

print("\nSliced_Seq length distribution:")
print(merged_df["Sliced_Seq_len"].value_counts().sort_index())

In [ ]:
new_df = merged_df.drop(columns=["HLA_Seq", "start_idx", "end_idx", "Sliced_Seq","HLA_Seq_len", "Sliced_Seq_len" ])
new_df = new_df.rename(columns={"tmp": "HLA_Seq"})
new_df = new_df.sort_values(by=["HLA_Name"])
new_df = new_df.reset_index(drop=True)

new_df.to_csv("HLA2_IMGT_MSA_sliced_idx_edit.csv", index=False)
new_df

In [ ]:
# rows whose HLA_Seq is 285 residues
hla_285 = merged_df[merged_df["HLA_Seq_len"] == 285]
hla_285 = hla_285.reset_index(drop=True)
hla_285

In [ ]:
# rows whose Sliced_Seq is 82 residues
sliced_82 = merged_df[merged_df["Sliced_Seq_len"] == 82]
# sliced_82 = sliced_82.sort_values(by=["HLA_Name"])
# sliced_82 = sliced_82.reset_index(drop=True)
sliced_82